In [1]:
import os
from pyspark.sql import SparkSession
import pandas as pd

# 1. CẤU HÌNH PACKAGES
os.environ['PYSPARK_SUBMIT_ARGS'] = (
    '--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.1,'
    'org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.3.1,'
    'org.projectnessie.nessie-integrations:nessie-spark-extensions-3.4_2.12:0.67.0,'
    'org.apache.hadoop:hadoop-aws:3.3.4 '
    'pyspark-shell'
)

# 2. KHỞI TẠO SPARK
spark = SparkSession.builder \
    .appName("Check-Gold-Data-Pandas") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.spark.extensions.NessieSparkSessionExtensions") \
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v1") \
    .config("spark.sql.catalog.nessie.ref", "main") \
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog") \
    .config("spark.sql.catalog.nessie.warehouse", "s3a://gold/") \
    .config("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()



In [2]:
# 3. LẤY DỮ LIỆU VÀ IN BÁO CÁO
gold_tables = ["dim_customer", "dim_location", "dim_product", "dim_time", "fact_order"]
report_data = []

print("\n🚀 Đang quét dữ liệu từ Iceberg (Nessie Catalog)...")

for table in gold_tables:
    try:
        # Lấy count và 1 dòng mẫu để kiểm tra schema
        count = spark.table(f"nessie.gold_db.{table}").count()
        report_data.append({"Table Name": table, "Record Count": count, "Status": "✅ OK"})
    except Exception as e:
        report_data.append({"Table Name": table, "Record Count": 0, "Status": f"❌ Error: {str(e)[:30]}"})

# 4. DÙNG PANDAS ĐỂ HIỂN THỊ
df_report = pd.DataFrame(report_data)

print("\n" + "="*60)
print("📊 TỔNG HỢP DỮ LIỆU TẦNG GOLD (MEDALLION ARCHITECTURE)")
print("="*60)
print(df_report.to_string(index=False)) # to_string giúp in bảng đẹp mà không bị mất cột
print("="*60)




🚀 Đang quét dữ liệu từ Iceberg (Nessie Catalog)...

📊 TỔNG HỢP DỮ LIỆU TẦNG GOLD (MEDALLION ARCHITECTURE)
  Table Name  Record Count Status
dim_customer          5027   ✅ OK
dim_location            52   ✅ OK
 dim_product        857821   ✅ OK
    dim_time          1912   ✅ OK
  fact_order       1670103   ✅ OK


In [3]:
# 5. IN THỬ DỮ LIỆU CHI TIẾT CỦA BẢNG FACT (5 dòng đầu)
print("\n🔍 CHI TIẾT 5 DÒNG MỚI NHẤT TRONG FACT_ORDER:")
try:
    # Chuyển 5 dòng sang Pandas DataFrame để in cho đẹp
    fact_sample = spark.table("nessie.gold_db.fact_order").limit(5).toPandas()
    print(fact_sample.to_string())
except:
    print("Không thể lấy dữ liệu chi tiết của fact_order.")


🔍 CHI TIẾT 5 DÒNG MỚI NHẤT TRONG FACT_ORDER:
    time_id        customer_id  product_id  location_id  unit_price  quantity  total_revenue
0  20221126  R_262vJPpUeeFKuHT  0307118932  34359738369        3.99       1.0           3.99
1  20210419  R_2AWkL1j4mAcAll8  0307464970  34359738369       15.99       1.0          15.99
2  20200609  R_tMsVLA8C6RuTmlb  048640708X  34359738369        7.89       1.0           7.89
3  20191221  R_262vJPpUeeFKuHT  0812971833  34359738369       12.60       1.0          12.60
4  20221214  R_AmMCFCEBSSiRhLz  1492677698  34359738369       17.99       1.0          17.99
